In [1]:
import os
import pandas as pd
pd.options.display.float_format = '{:.3f}'.format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
import numpy as np
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB
from itertools import product
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from tqdm import tqdm
from functions_utils import *
from functions_data import *
from functions_optimize import *
from functions_eval import *

generation_data, I, T = load_generation_data(date_filter="2022-07-18")
S, R, P_RT, K, K0, M1, M2 = load_parameters(I, T, generation_data)
P_DA, P_PN = load_price_data()

BASE_PATH = "/Users/jangseohyun/SynologyDrive/workspace/symply/DER/opt_result"

✅ 총 10개 파일을 불러왔습니다: 1201.csv, 137.csv, 281.csv, 397.csv, 401.csv, 430.csv, 514.csv, 524.csv, 775.csv, 89.csv
📊 데이터 Shape: I=10, T=24, S=200
✅ 시뮬레이션 초기화 완료: S=200, Randomness='high', M1=773.00, M2=2545.00


In [2]:
def save_scenario_data(R, P_DA, P_RT, P_PN, I, T, S, base_dir=BASE_PATH):
    """
    시나리오 데이터(R, P_DA, P_RT, P_PN)를 CSV로 저장
    """
    save_dir = os.path.join(base_dir, f"i_{I}_s_{S}")
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    print(f"🔄 시나리오 데이터 저장 중... (폴더: {save_dir})")
    
    # 1. R: 발전량 시나리오 (I × T × S)
    R_data = []
    for i, t, s in product(range(I), range(T), range(S)):
        R_data.append({
            'i': i,
            't': t,
            's': s,
            'R': R[i, t, s]
        })
    pd.DataFrame(R_data).to_csv(f"{save_dir}/R.csv", index=False)
    print("✅ R.csv 저장 완료")
    
    # 2. P_DA: Day-Ahead 가격 (T,)
    P_DA_data = []
    for t in range(T):
        P_DA_data.append({
            't': t,
            'P_DA': P_DA[t]
        })
    pd.DataFrame(P_DA_data).to_csv(f"{save_dir}/P_DA.csv", index=False)
    print("✅ P_DA.csv 저장 완료")
    
    # 3. P_RT: Real-Time 가격 시나리오 (T × S)
    P_RT_data = []
    for t, s in product(range(T), range(S)):
        P_RT_data.append({
            't': t,
            's': s,
            'P_RT': P_RT[t, s]
        })
    pd.DataFrame(P_RT_data).to_csv(f"{save_dir}/P_RT.csv", index=False)
    print("✅ P_RT.csv 저장 완료")
    
    # 4. P_PN: Penalty 가격 (T,)
    P_PN_data = []
    for t in range(T):
        P_PN_data.append({
            't': t,
            'P_PN': P_PN[t]
        })
    pd.DataFrame(P_PN_data).to_csv(f"{save_dir}/P_PN.csv", index=False)
    print("✅ P_PN.csv 저장 완료")
    
    print(f"\n🎉 모든 시나리오 데이터가 '{save_dir}' 폴더에 저장되었습니다!")
    print(f"📁 총 4개 파일 생성")
    
    return save_dir

In [3]:
def load_scenario_data(I, T, S, base_dir=BASE_PATH):
    """
    저장된 시나리오 데이터를 불러오기
    """
    save_dir = os.path.join(base_dir, f"i_{I}_s_{S}")
    print(f"🔄 시나리오 데이터 불러오는 중... (폴더: {save_dir})")
    
    results = {}
    
    # 1. R 불러오기 (I × T × S)
    R_df = pd.read_csv(f"{save_dir}/R.csv")
    R = np.zeros((I, T, S))
    for _, row in R_df.iterrows():
        R[int(row['i']), int(row['t']), int(row['s'])] = row['R']
    results['R'] = R
    print("✅ R 불러오기 완료")
    
    # 2. P_DA 불러오기 (T,)
    P_DA_df = pd.read_csv(f"{save_dir}/P_DA.csv")
    P_DA = np.zeros(T)
    for _, row in P_DA_df.iterrows():
        P_DA[int(row['t'])] = row['P_DA']
    results['P_DA'] = P_DA
    print("✅ P_DA 불러오기 완료")
    
    # 3. P_RT 불러오기 (T × S)
    P_RT_df = pd.read_csv(f"{save_dir}/P_RT.csv")
    P_RT = np.zeros((T, S))
    for _, row in P_RT_df.iterrows():
        P_RT[int(row['t']), int(row['s'])] = row['P_RT']
    results['P_RT'] = P_RT
    print("✅ P_RT 불러오기 완료")
    
    # 4. P_PN 불러오기 (T,)
    P_PN_df = pd.read_csv(f"{save_dir}/P_PN.csv")
    P_PN = np.zeros(T)
    for _, row in P_PN_df.iterrows():
        P_PN[int(row['t'])] = row['P_PN']
    results['P_PN'] = P_PN
    print("✅ P_PN 불러오기 완료")
    
    print(f"\n🎉 모든 시나리오 데이터 불러오기 완료!")
    return results 

In [4]:
def save_holistic_results(x_hol, a_hol, yp_hol, ym_hol, z_hol, zc_hol, zd_hol, 
                               ep_hol, bp_hol, em_hol, bm_hol, d_hol, dp_hol, dm_hol, 
                               obj_hol, I, T, S, base_dir=BASE_PATH):
    
    # 폴더 구조 생성: opt_result/i_{I}_s_{S}/
    save_dir = os.path.join(base_dir, f"i_{I}_s_{S}")
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    print(f"🔄 최적화 결과 저장 중... (폴더: {save_dir})")
    
    # 1. x_hol: 발전량 (I × T)
    x_data = []
    for i, t in product(range(I), range(T)):
        x_data.append({
            'i': i,
            't': t,
            'x_hol': x_hol[i, t]
        })
    pd.DataFrame(x_data).to_csv(f"{save_dir}/x_hol.csv", index=False)
    print("✅ x_hol.csv 저장 완료")
    
    # 2. a_hol: 총 발전량 (T,)
    a_data = []
    for t in range(T):
        a_data.append({
            't': t,
            'a_hol': a_hol[t]
        })
    pd.DataFrame(a_data).to_csv(f"{save_dir}/a_hol.csv", index=False)
    print("✅ a_hol.csv 저장 완료")
    
    # 3. yp_hol: 양의 편차 (I × T × S)
    yp_data = []
    for i, t, s in product(range(I), range(T), range(S)):
        yp_data.append({
            'i': i,
            't': t,
            's': s,
            'yp_hol': yp_hol[i, t, s]
        })
    pd.DataFrame(yp_data).to_csv(f"{save_dir}/yp_hol.csv", index=False)
    print("✅ yp_hol.csv 저장 완료")
    
    # 4. ym_hol: 음의 편차 (I × T × S)
    ym_data = []
    for i, t, s in product(range(I), range(T), range(S)):
        ym_data.append({
            'i': i,
            't': t,
            's': s,
            'ym_hol': ym_hol[i, t, s]
        })
    pd.DataFrame(ym_data).to_csv(f"{save_dir}/ym_hol.csv", index=False)
    print("✅ ym_hol.csv 저장 완료")
    
    # 5. z_hol: 배터리 상태 (I × T × S)
    z_data = []
    for i, t, s in product(range(I), range(T), range(S)):
        z_data.append({
            'i': i,
            't': t,
            's': s,
            'z_hol': z_hol[i, t, s]
        })
    pd.DataFrame(z_data).to_csv(f"{save_dir}/z_hol.csv", index=False)
    print("✅ z_hol.csv 저장 완료")
    
    # 6. zc_hol: 배터리 충전 (I × T × S)
    zc_data = []
    for i, t, s in product(range(I), range(T), range(S)):
        zc_data.append({
            'i': i,
            't': t,
            's': s,
            'zc_hol': zc_hol[i, t, s]
        })
    pd.DataFrame(zc_data).to_csv(f"{save_dir}/zc_hol.csv", index=False)
    print("✅ zc_hol.csv 저장 완료")
    
    # 7. zd_hol: 배터리 방전 (I × T × S)
    zd_data = []
    for i, t, s in product(range(I), range(T), range(S)):
        zd_data.append({
            'i': i,
            't': t,
            's': s,
            'zd_hol': zd_hol[i, t, s]
        })
    pd.DataFrame(zd_data).to_csv(f"{save_dir}/zd_hol.csv", index=False)
    print("✅ zd_hol.csv 저장 완료")
    
    # 8. ep_hol: 에너지 양의 편차 (I × T × S)
    ep_data = []
    for i, t, s in product(range(I), range(T), range(S)):
        ep_data.append({
            'i': i,
            't': t,
            's': s,
            'ep_hol': ep_hol[i, t, s]
        })
    pd.DataFrame(ep_data).to_csv(f"{save_dir}/ep_hol.csv", index=False)
    print("✅ ep_hol.csv 저장 완료")
    
    # 9. bp_hol: 총 에너지 양의 편차 (T × S)
    bp_data = []
    for t, s in product(range(T), range(S)):
        bp_data.append({
            't': t,
            's': s,
            'bp_hol': bp_hol[t, s]
        })
    pd.DataFrame(bp_data).to_csv(f"{save_dir}/bp_hol.csv", index=False)
    print("✅ bp_hol.csv 저장 완료")
    
    # 10. em_hol: 에너지 음의 편차 (I × T × S)
    em_data = []
    for i, t, s in product(range(I), range(T), range(S)):
        em_data.append({
            'i': i,
            't': t,
            's': s,
            'em_hol': em_hol[i, t, s]
        })
    pd.DataFrame(em_data).to_csv(f"{save_dir}/em_hol.csv", index=False)
    print("✅ em_hol.csv 저장 완료")
    
    # 11. bm_hol: 총 에너지 음의 편차 (T × S)
    bm_data = []
    for t, s in product(range(T), range(S)):
        bm_data.append({
            't': t,
            's': s,
            'bm_hol': bm_hol[t, s]
        })
    pd.DataFrame(bm_data).to_csv(f"{save_dir}/bm_hol.csv", index=False)
    print("✅ bm_hol.csv 저장 완료")
    
    # 12. d_hol: 교환량 (I × I × T × S) - i와 j로 구분
    d_data = []
    for i, j, t, s in product(range(I), range(I), range(T), range(S)):
        if i != j:
            d_data.append({
                'i': i,
                'j': j,
                't': t,
                's': s,
                'd_hol': d_hol[i, j, t, s]
            })
    pd.DataFrame(d_data).to_csv(f"{save_dir}/d_hol.csv", index=False)
    print("✅ d_hol.csv 저장 완료")
    
    # 13. dp_hol: 발전소별 총 송출량 (I × T × S)
    dp_data = []
    for i, t, s in product(range(I), range(T), range(S)):
        dp_data.append({
            'i': i,
            't': t,
            's': s,
            'dp_hol': dp_hol[i, t, s]
        })
    pd.DataFrame(dp_data).to_csv(f"{save_dir}/dp_hol.csv", index=False)
    print("✅ dp_hol.csv 저장 완료")
    
    # 14. dm_hol: 발전소별 총 수신량 (I × T × S)
    dm_data = []
    for i, t, s in product(range(I), range(T), range(S)):
        dm_data.append({
            'i': i,
            't': t,
            's': s,
            'dm_hol': dm_hol[i, t, s]
        })
    pd.DataFrame(dm_data).to_csv(f"{save_dir}/dm_hol.csv", index=False)
    print("✅ dm_hol.csv 저장 완료")
    
    # 15. obj_hol: 목적함수 값 (스칼라)
    obj_data = [{
        'obj_hol': obj_hol,
        'I': I,
        'T': T,
        'S': S
    }]
    pd.DataFrame(obj_data).to_csv(f"{save_dir}/obj_hol.csv", index=False)
    print("✅ obj_hol.csv 저장 완료")
    
    print(f"\n🎉 모든 변수가 '{save_dir}' 폴더에 저장되었습니다!")
    print(f"📁 총 {15}개 파일 생성")
    
    return save_dir

In [5]:
def load_holistic_results(I, T, S, base_dir=BASE_PATH):
    save_dir = os.path.join(base_dir, f"i_{I}_s_{S}")
    print(f"🔄 최적화 결과 불러오는 중... (폴더: {save_dir})")
    
    results = {}
    
    # 1. x_hol 불러오기 (I × T)
    x_df = pd.read_csv(f"{save_dir}/x_hol.csv")
    x_hol = np.zeros((I, T))
    for _, row in x_df.iterrows():
        x_hol[int(row['i']), int(row['t'])] = row['x_hol']
    results['x_hol'] = x_hol
    print("✅ x_hol 불러오기 완료")
    
    # 2. a_hol 불러오기 (T,)
    a_df = pd.read_csv(f"{save_dir}/a_hol.csv")
    a_hol = np.zeros(T)
    for _, row in a_df.iterrows():
        a_hol[int(row['t'])] = row['a_hol']
    results['a_hol'] = a_hol
    print("✅ a_hol 불러오기 완료")
    
    # 3. yp_hol 불러오기 (I × T × S)
    yp_df = pd.read_csv(f"{save_dir}/yp_hol.csv")
    yp_hol = np.zeros((I, T, S))
    for _, row in yp_df.iterrows():
        yp_hol[int(row['i']), int(row['t']), int(row['s'])] = row['yp_hol']
    results['yp_hol'] = yp_hol
    print("✅ yp_hol 불러오기 완료")
    
    # 4. ym_hol 불러오기 (I × T × S)
    ym_df = pd.read_csv(f"{save_dir}/ym_hol.csv")
    ym_hol = np.zeros((I, T, S))
    for _, row in ym_df.iterrows():
        ym_hol[int(row['i']), int(row['t']), int(row['s'])] = row['ym_hol']
    results['ym_hol'] = ym_hol
    print("✅ ym_hol 불러오기 완료")
    
    # 5. z_hol 불러오기 (I × T × S)
    z_df = pd.read_csv(f"{save_dir}/z_hol.csv")
    z_hol = np.zeros((I, T, S))
    for _, row in z_df.iterrows():
        z_hol[int(row['i']), int(row['t']), int(row['s'])] = row['z_hol']
    results['z_hol'] = z_hol
    print("✅ z_hol 불러오기 완료")
    
    # 6. zc_hol 불러오기 (I × T × S)
    zc_df = pd.read_csv(f"{save_dir}/zc_hol.csv")
    zc_hol = np.zeros((I, T, S))
    for _, row in zc_df.iterrows():
        zc_hol[int(row['i']), int(row['t']), int(row['s'])] = row['zc_hol']
    results['zc_hol'] = zc_hol
    print("✅ zc_hol 불러오기 완료")
    
    # 7. zd_hol 불러오기 (I × T × S)
    zd_df = pd.read_csv(f"{save_dir}/zd_hol.csv")
    zd_hol = np.zeros((I, T, S))
    for _, row in zd_df.iterrows():
        zd_hol[int(row['i']), int(row['t']), int(row['s'])] = row['zd_hol']
    results['zd_hol'] = zd_hol
    print("✅ zd_hol 불러오기 완료")
    
    # 8. ep_hol 불러오기 (I × T × S)
    ep_df = pd.read_csv(f"{save_dir}/ep_hol.csv")
    ep_hol = np.zeros((I, T, S))
    for _, row in ep_df.iterrows():
        ep_hol[int(row['i']), int(row['t']), int(row['s'])] = row['ep_hol']
    results['ep_hol'] = ep_hol
    print("✅ ep_hol 불러오기 완료")
    
    # 9. bp_hol 불러오기 (T × S)
    bp_df = pd.read_csv(f"{save_dir}/bp_hol.csv")
    bp_hol = np.zeros((T, S))
    for _, row in bp_df.iterrows():
        bp_hol[int(row['t']), int(row['s'])] = row['bp_hol']
    results['bp_hol'] = bp_hol
    print("✅ bp_hol 불러오기 완료")
    
    # 10. em_hol 불러오기 (I × T × S)
    em_df = pd.read_csv(f"{save_dir}/em_hol.csv")
    em_hol = np.zeros((I, T, S))
    for _, row in em_df.iterrows():
        em_hol[int(row['i']), int(row['t']), int(row['s'])] = row['em_hol']
    results['em_hol'] = em_hol
    print("✅ em_hol 불러오기 완료")
    
    # 11. bm_hol 불러오기 (T × S)
    bm_df = pd.read_csv(f"{save_dir}/bm_hol.csv")
    bm_hol = np.zeros((T, S))
    for _, row in bm_df.iterrows():
        bm_hol[int(row['t']), int(row['s'])] = row['bm_hol']
    results['bm_hol'] = bm_hol
    print("✅ bm_hol 불러오기 완료")
    
    # 12. d_hol 불러오기 (I × I × T × S)
    d_df = pd.read_csv(f"{save_dir}/d_hol.csv")
    d_hol = np.zeros((I, I, T, S))
    for _, row in d_df.iterrows():
        d_hol[int(row['i']), int(row['j']), int(row['t']), int(row['s'])] = row['d_hol']
    results['d_hol'] = d_hol
    print("✅ d_hol 불러오기 완료")
    
    # 13. dp_hol 불러오기 (I × T × S)
    dp_df = pd.read_csv(f"{save_dir}/dp_hol.csv")
    dp_hol = np.zeros((I, T, S))
    for _, row in dp_df.iterrows():
        dp_hol[int(row['i']), int(row['t']), int(row['s'])] = row['dp_hol']
    results['dp_hol'] = dp_hol
    print("✅ dp_hol 불러오기 완료")
    
    # 14. dm_hol 불러오기 (I × T × S)
    dm_df = pd.read_csv(f"{save_dir}/dm_hol.csv")
    dm_hol = np.zeros((I, T, S))
    for _, row in dm_df.iterrows():
        dm_hol[int(row['i']), int(row['t']), int(row['s'])] = row['dm_hol']
    results['dm_hol'] = dm_hol
    print("✅ dm_hol 불러오기 완료")
    
    # 15. obj_hol 불러오기 (스칼라)
    obj_df = pd.read_csv(f"{save_dir}/obj_hol.csv")
    obj_hol = obj_df['obj_hol'].iloc[0]
    results['obj_hol'] = obj_hol
    print("✅ obj_hol 불러오기 완료")
    
    print(f"\n🎉 모든 변수 불러오기 완료!")
    return results

In [6]:
def check_files_exist(I, S, base_dir=BASE_PATH):
    """
    특정 I, S 조합의 폴더에 모든 CSV 파일이 존재하는지 확인
    """
    save_dir = os.path.join(base_dir, f"i_{I}_s_{S}")
    if not os.path.exists(save_dir):
        return False, f"폴더가 존재하지 않습니다: {save_dir}"
    
    required_files = [
        'x_hol.csv', 'a_hol.csv', 'yp_hol.csv', 'ym_hol.csv', 'z_hol.csv',
        'zc_hol.csv', 'zd_hol.csv', 'ep_hol.csv', 'bp_hol.csv', 'em_hol.csv',
        'bm_hol.csv', 'd_hol.csv', 'dp_hol.csv', 'dm_hol.csv', 'obj_hol.csv',
        'R.csv', 'P_DA.csv', 'P_RT.csv', 'P_PN.csv'
    ]
    
    missing_files = []
    for file in required_files:
        if not os.path.exists(os.path.join(save_dir, file)):
            missing_files.append(file)
    
    if missing_files:
        return False, f"누락된 파일: {missing_files}"
    else:
        return True, "모든 파일이 존재합니다"

In [ ]:
print("[Holistic Aggregation Model optimization]")

# 시나리오 데이터 저장
save_scenario_data(R, P_DA, P_RT, P_PN, I, T, S, base_dir=BASE_PATH)

# 최적화 실행
x_hol, a_hol, yp_hol, ym_hol, z_hol, zc_hol, zd_hol, ep_hol, bp_hol, em_hol, bm_hol, d_hol, dp_hol, dm_hol, obj_hol = optimize_hol(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1, M2)
print("-" * 100)

# 결과 저장
save_dir = save_holistic_results(
    x_hol, a_hol, yp_hol, ym_hol, z_hol, zc_hol, zd_hol, 
    ep_hol, bp_hol, em_hol, bm_hol, d_hol, dp_hol, dm_hol, 
    obj_hol, I, T, S, base_dir=BASE_PATH
)

# 파일 존재 여부 확인
exists, message = check_files_exist(I, S, base_dir=BASE_PATH)
print(f"파일 체크: {message}")

# 불러오기
loaded_results = load_holistic_results(I, T, S, base_dir=BASE_PATH)

# 최적화 결과 불러오기
x = loaded_results['x_hol']
a = loaded_results['a_hol']
yp = loaded_results['yp_hol']
ym = loaded_results['ym_hol']
z = loaded_results['z_hol']
zc = loaded_results['zc_hol']
zd = loaded_results['zd_hol']
ep = loaded_results['ep_hol']
bp = loaded_results['bp_hol']
em = loaded_results['em_hol']
bm = loaded_results['bm_hol']
d = loaded_results['d_hol']
dp = loaded_results['dp_hol']
dm = loaded_results['dm_hol']
obj = loaded_results['obj_hol']

# 시나리오 데이터도 불러오기
scenario_data = load_scenario_data(I, T, S, base_dir=BASE_PATH)
R = scenario_data['R']
P_DA = scenario_data['P_DA']
P_RT = scenario_data['P_RT']
P_PN = scenario_data['P_PN']

[Holistic Aggregation Model optimization]
🔄 시나리오 데이터 저장 중... (폴더: /Users/jangseohyun/SynologyDrive/workspace/symply/DER/opt_result\i_10_s_200)
✅ R.csv 저장 완료
✅ P_DA.csv 저장 완료
✅ P_RT.csv 저장 완료
✅ P_PN.csv 저장 완료

🎉 모든 시나리오 데이터가 '/Users/jangseohyun/SynologyDrive/workspace/symply/DER/opt_result\i_10_s_200' 폴더에 저장되었습니다!
📁 총 4개 파일 생성
Set parameter Username
Set parameter LicenseID to value 2681721
Academic license - for non-commercial use only - expires 2026-06-24
Set parameter MIPGap to value 1e-07
